# Explore the data

`%%bq` runs SQL through the lab's guard: it dry-runs first and refuses a query that would scan more than the cap (raise it with `--max-gb`), caps the result
size, and caches the answer on disk (`--refresh` to re-run). The result lands in a DataFrame named by the last word on the line.

In [ ]:
%load_ext mllab.magics

What is in the 2026 season dataset, and how big is it? (Table metadata, so this costs almost nothing.)

In [ ]:
%%bq --max-gb 0.1 tables
SELECT table_id, row_count, ROUND(size_bytes / 1e6, 1) AS mb
FROM `hankstank.mlb_2026_season.__TABLES__`
ORDER BY table_id

In [ ]:
tables

## How have the live predictions done?

One row per completed game: the **last prediction made before first pitch**, and the result. This uses the same rules as the lab's own backtest data
(`sql/mlb_v10_pregame.sql`): predictions made after first pitch are contaminated, and a game can have a skeleton row plus a final row.

In [ ]:
%%bq --max-gb 1 scored
WITH pred AS (
  SELECT game_pk, home_win_probability AS p
  FROM `hankstank.mlb_2026_season.game_predictions`
  WHERE predicted_at < game_time_utc
  QUALIFY ROW_NUMBER() OVER (PARTITION BY game_pk ORDER BY predicted_at DESC) = 1
), final AS (
  SELECT game_pk, CAST(home_score > away_score AS INT64) AS home_win
  FROM `hankstank.mlb_2026_season.games`
  WHERE status IN ('Final', 'Game Over') AND home_score IS NOT NULL AND away_score IS NOT NULL AND home_score != away_score
  QUALIFY ROW_NUMBER() OVER (PARTITION BY game_pk ORDER BY synced_at DESC NULLS LAST) = 1
)
SELECT p, home_win FROM pred JOIN final USING (game_pk)

In [ ]:
import numpy as np
n = len(scored)
acc = float(((scored.p > 0.5) == (scored.home_win == 1)).mean())
ll = float(-np.mean(scored.home_win * np.log(scored.p.clip(1e-6, 1 - 1e-6)) + (1 - scored.home_win) * np.log((1 - scored.p).clip(1e-6, 1 - 1e-6))))
base = float(scored.home_win.mean())
print(f"{n} completed games with a pre-game prediction")
print(f"accuracy {acc:.3f}   log loss {ll:.4f}   (always-home: accuracy {max(base, 1 - base):.3f})")

**Calibration**: when the model says 60%, does the home team win about 60% of the time?

In [ ]:
import pandas as pd
bins = pd.cut(scored.p, [0, .40, .45, .50, .55, .60, .65, 1.0])
cal = scored.groupby(bins, observed=True).agg(games=("p", "size"), predicted=("p", "mean"), actual=("home_win", "mean")).round(3)
cal

A few hundred games per bin is the least you need before a gap means anything. With about 2,000 games in total, only differences of two or three accuracy points
are detectable at all, which is why the next notebook reports confidence intervals rather than a single number.

**Next:** `02_backtest_models`.